# Bibliotecas

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import yaml
import cv2

from tqdm import tqdm
from ultralytics import YOLO

# Baixando Arquivos

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aklimarimi/8-facial-expressions-for-yolo")

print("Path to dataset files:", path)

# Verificação da Estrutura de Pastas

In [ ]:
base_path = r"C:\Users\Maiquel\.cache\kagglehub\datasets\aklimarimi\8-facial-expressions-for-yolo\versions\4"

def list_structure(startpath):
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/ - ({len(files)} arquivos)')

print("Estrutura do Dataset:")
list_structure(base_path)

# Definindo os Caminhos Corretos

In [ ]:
BASE_PATH = r"C:\Users\Maiquel\.cache\kagglehub\datasets\aklimarimi\8-facial-expressions-for-yolo\versions\4/9 Facial Expressions you need"

# Subpastas
train_path = os.path.join(BASE_PATH, "train")
val_path = os.path.join(BASE_PATH, "valid")
test_path = os.path.join(BASE_PATH, "test")

# Mapeamento de 9 classes
class_map = {
    0: 'Angry', 1: 'Contempt', 2: 'Disgust', 3: 'Fear',
    4: 'Happy', 5: 'Natural', 6: 'Sad', 7: 'Sleepy', 8: 'Surprised'
}

# Análise de Desbalanceamento

In [ ]:
def get_class_distribution(label_dir):
    counts = []
    files = [f for f in os.listdir(label_dir) if f.endswith('.txt')]
    for file in tqdm(files, desc=f"Lendo {label_dir}"):
        with open(os.path.join(label_dir, file), 'r') as f:
            for line in f:
                class_id = int(line.split()[0])
                counts.append(class_map.get(class_id, f"Unknown({class_id})"))
    return counts


train_labels = get_class_distribution(os.path.join(train_path, "labels"))
df_dist = pd.DataFrame(train_labels, columns=['Emotion'])


plt.figure(figsize=(12, 6))
sns.countplot(data=df_dist, x='Emotion', palette='magma', order=list(class_map.values()))
plt.title('Projeto DEF - Distribuição de Classes no Treino')
plt.xticks(rotation=45)
plt.show()

print("\nContagem exata por classe:")
print(df_dist['Emotion'].value_counts())

# Criação do Arquivo de Configuração (data.yaml)

In [ ]:
# Criando o dicionário de configuração
config = {
    'train': os.path.join(train_path, 'images'),
    'val': os.path.join(val_path, 'images'),
    'test': os.path.join(test_path, 'images'),
    'nc': 9,
    'names': list(class_map.values())
}

# Salvando o arquivo .yaml
with open('projeto_def.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ Arquivo projeto_def.yaml criado!")

# Validação e Limpeza Final

In [ ]:
def sync_data(path):
    imgs = {os.path.splitext(f)[0] for f in os.listdir(os.path.join(path, "images"))}
    lbls = {os.path.splitext(f)[0] for f in os.listdir(os.path.join(path, "labels"))}
    orphans = lbls - imgs
    for label in orphans:
        os.remove(os.path.join(path, "labels", label + ".txt"))
    print(f"Sincronizado: {path} (Removidos {len(orphans)} arquivos órfãos)")

sync_data(train_path)

# Treinando o Modelo

In [ ]:
# Carrega um modelo pré-treinado (ajuda a IA a aprender mais rápido)
model = YOLO('yolov8n.pt')

# Inicia o treinamento
results = model.train(
    data='projeto_def.yaml', # O arquivo que criamos no passo 1
    epochs=20,               # Quantas vezes ele vai ler todo o dataset
    imgsz=640,               # Tamanho da imagem
    batch=16,                # Quantas imagens processa por vez (ajuste se der erro de memória)
    device=0,                # Garante que está na GPU
    workers=4,               # Usa mais núcleos do seu processador para carregar as fotos
    name='treino_def_v1'     # Nome da pasta onde salvará os resultados
)

# Avaliando o Modelo

In [ ]:
Modelo = r'C:\Users\Maiquel\runs\detect\treino_def_v15\weights\best.pt'

print(f"✅ Utilizando o modelo treinado em: {Modelo}")


model = YOLO(Modelo)
print("🚀 Modelo pronto para uso!")

In [ ]:
metrics = model.val()

classes = metrics.names
map50_values = metrics.box.maps

print("📊 DESEMPENHO POR EXPRESSÃO FACIAL (Em % de acerto):")
print("-" * 45)

for i, valor in enumerate(map50_values):
    nome_classe = classes[i]
    porcentagem = valor * 100
    print(f"🔹 {nome_classe:15} : {porcentagem:>6.2f}%")

print("-" * 45)
print(f"⭐ MÉDIA GERAL (mAP50): {metrics.box.map50 * 100:.2f}%")

In [ ]:
dados_finais = {
    'Happy': 77.08,
    'Disgust': 72.34,
    'Fear': 69.88,
    'Surprised': 69.21,
    'Angry': 68.54,
    'Sad': 65.01,
    'Contempt': 60.63,
    'Sleepy': 60.56,
    'Natural': 52.46
}

nomes = list(dados_finais.keys())[::-1]
valores = list(dados_finais.values())[::-1]

plt.figure(figsize=(11, 7))
cores = ['#2ecc71' if v > 75 else '#3498db' if v > 60 else '#e74c3c' for v in valores]
barras = plt.barh(nomes, valores, color=cores)

for i, v in enumerate(valores):
    plt.text(v + 1, i, f"{v}%", va='center', fontsize=11, fontweight='bold')

plt.xlabel('Precisão Real (mAP50-95) em %', fontsize=12)
plt.title('Performance do Modelo por Expressão Facial - Avaliação V1', fontsize=14, fontweight='bold')
plt.xlim(0, 100)
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()

plt.savefig('analise_performance_manual.png', dpi=300)
plt.show()

# Testando o Modelo JS

In [ ]:
# Usa a sua variável definida
model = YOLO(Modelo)

# Exporta para ONNX (formato para JavaScript/Web)
model.export(format='onnx')
print("✅ Arquivo 'best.onnx' gerado na mesma pasta do seu modelo.")

In [ ]:
model = YOLO(Modelo)

# Inicia a webcam
cap = cv2.VideoCapture(0)

print("🚀 Câmera aberta! Procure a janela 'Detector de Expressões' na sua barra de tarefas.")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break


    # persist=True ajuda a manter a "notificação" estável entre os frames
    results = model.track(frame, persist=True, conf=0.5)

    # Desenha os resultados no frame (Isso gera a "notificação" visual)
    annotated_frame = results[0].plot()

    # Exibe a janela com o texto da expressão
    cv2.imshow("Detector de Expressoes", annotated_frame)

    # Aperte 'q' para fechar
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# **Resumo do Projeto: Sistema de Detecção de Expressões Faciais (DEF) com YOLOv8**
* **Descrição Geral**: Projeto de caráter **acadêmico e prático** com foco no entendimento do **fluxo completo da engenharia de Deep Learning**
* **Objetivo Principal**: Aprender o processo **ponta a ponta**, desde **extração e tratamento de dados** até **deploy em tempo real**
* **Escopo do Projeto**: Não teve como meta um modelo **perfeito para produção**, mas sim um **laboratório real de aprendizado**
* **Resultado Final**: Modelo funcional executando detecção de emoções via **webcam em tempo real**

# **1. Natureza e Objetivo do Projeto**
* **Tipo de Projeto**: Iniciativa **acadêmica** com aplicação **prática**
* **Objetivo Central**: Dominar o **passo a passo da engenharia de Machine Learning**
* **Fluxo Trabalhado**: **Coleta de dados**, **pré-processamento**, **treinamento**, **validação** e **deploy**
* **Ambiente de Execução**: Treinamento local considerando **limitações de hardware**

# **2. Estratégia de Treinamento: Transfer Learning (Fine-Tuning)**
* **Abordagem Utilizada**: Uso de **Fine-Tuning** a partir do modelo base `yolov8n.pt`
* **Motivação Técnica**: Redução de **tempo de treinamento** e **uso de recursos computacionais**
* **Tempo de Treino**: Aproximadamente **4 horas**
* **Modelo Pré-Treinado**: Rede treinada previamente pela **Ultralytics** com **milhões de imagens**
* **Conhecimento Herdado**: Capacidade de reconhecer **bordas**, **texturas** e **rostos humanos**
* **Foco do Projeto**: Especializar o modelo para diferenciar **9 emoções faciais** do dataset

# **3. Pré-processamento e Limpeza de Dados**
* **Objetivo do Pré-processamento**: Evitar **falhas durante o treinamento**
* **Análise de Distribuição**: Avaliação do **balanceamento das classes**
* **Resultado da Análise**: Identificação de maior volume de imagens da classe **Happy**
* **Sincronização de Arquivos**: Desenvolvimento de script em **Python** para cruzar imagens e arquivos `.txt`
* **Limpeza de Dados**: Remoção de imagens **órfãs** sem marcação
* **Benefícios Técnicos**: Prevenção de **travamentos**, **OOM** e **dados inconsistentes**

# **4. Resultados Obtidos: Um Olhar Sincero**
* **Contexto do Modelo**: Uso do modelo **Nano (`yolov8n`)** em dataset **complexo**
* **Desempenho Positivo**: Classes **Happy** e **Sleepy** com **alta precisão**
* **Motivo do Bom Resultado**: Expressões com **traços faciais bem definidos**
* **Limitações do Modelo**: Dificuldade em reconhecer **expressões sutis**
* **Classe com Baixo Desempenho**: **Natural (rosto neutro)**
* **Erros Comuns**: Confusão entre **Natural**, **Contempt** e **Sad**
* **Conclusão Técnica**: **Microexpressões** exigem modelos mais **robustos** para evitar **falsos positivos**

# **5. Aplicações Práticas (Casos de Uso)**
* **Segurança no Trânsito**: Detecção da classe **Sleepy** para alertar motoristas sonolentos
* **Pesquisa e Varejo**: Análise de reações **Happy**, **Surprised** e **Angry** em lojas físicas
* **Coleta de Feedback**: Substituição de pesquisas manuais por dados **orgânicos**
* **Acessibilidade**: Apoio a pessoas com **TEA** na interpretação de **emoções sociais**

# **6. Possíveis Melhorias do Projeto**
* **Evolução do Modelo**: Migração para o modelo `yolov8s`
* **Capacidade Técnica**: Melhor captura de **detalhes sutis** das expressões faciais
* **Equilíbrio de Recursos**: Aumento de **precisão** mantendo consumo de memória aceitável
* **Early Stopping**: Parada automática do treinamento ao não detectar **melhorias**
* **Otimização Computacional**: Economia de **tempo** e **poder de processamento**
* **Ajustes de Iluminação**: Simulação de **variações de luz e cor** via hiperparâmetros
* **Robustez Final**: Preparação do modelo para **webcams reais** e ambientes imprevisíveis

# **Referências**
* **Dataset Utilizado**: [8 Facial Expressions for YOLO — Kaggle](https://www.kaggle.com/datasets/aklimarimi/8-facial-expressions-for-yolo)
